# Zomato Delivery Time Prediction — ML Model

**Dataset:** Zomato Delivery Operations (45,584 rows)  
**Target:** `Time_taken (min)`

| Model | MAE | RMSE | R² |
|---|---|---|---|
| Random Forest | 2.97 min | 3.69 min | 0.846 |
| Gradient Boost | 3.06 min | 3.81 min | 0.836 |
| Ridge Regression | 5.01 min | 6.27 min | 0.557 |

**CV-5 RF MAE:** 2.98 ± 0.017

**Top features (RF importance):**
- Rider rating: 23.1%
- Traffic density: 15.8%
- Weather: 15.2%
- Distance (km): 14.6% ← engineered
- Multiple trips: 12.6%
- Rider age: 9.1%
- Vehicle condition: 7.6%

## 0. Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

## 1. Encoding Maps & Feature Definitions

In [ ]:
# Encoding maps
WEATHER_MAP  = {"Sunny": 0, "Windy": 1, "Sandstorms": 2,
                "Stormy": 3, "Fog": 4, "Cloudy": 5}
TRAFFIC_MAP  = {"Low": 0, "Medium": 1, "High": 2, "Jam": 3}
VEHICLE_MAP  = {"electric_scooter": 0, "scooter": 1,
                "bicycle": 2, "motorcycle": 3}
CITY_MAP     = {"Urban": 0, "Metropolitian": 1, "Semi-Urban": 2}
ORDER_MAP    = {"Snack": 0, "Drinks": 0, "Meal": 1, "Buffet": 2}
FESTIVAL_MAP = {"No": 0, "Yes": 1}

BASE_FEATURES = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Vehicle_condition",
    "multiple_deliveries",
    "weather_n",
    "traffic_n",
    "vehicle_n",
    "city_n",
    "order_n",
    "festival_n",
    "distance_km",
    "pickup_delay_min",
]

FEATURE_LABELS = {
    "Delivery_person_Age":     "Rider age",
    "Delivery_person_Ratings": "Rider rating",
    "Vehicle_condition":       "Vehicle condition",
    "multiple_deliveries":     "Multiple deliveries",
    "weather_n":               "Weather conditions",
    "traffic_n":               "Traffic density",
    "vehicle_n":               "Vehicle type",
    "city_n":                  "City type",
    "order_n":                 "Order type",
    "festival_n":              "Festival flag",
    "distance_km":             "Distance (km)",
    "pickup_delay_min":        "Pickup delay (min)",
}

## 2. Data Loading

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load and do basic type-cleaning on the raw delivery CSV."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    # Strip whitespace from string columns
    str_cols = df.select_dtypes("object").columns
    df[str_cols] = df[str_cols].apply(lambda c: c.str.strip())

    print(f"[load]  Rows loaded   : {len(df):,}")
    print(f"[load]  Columns       : {df.columns.tolist()}")
    return df

## 3. Feature Engineering

In [ ]:
def _haversine_km(lat1: pd.Series, lon1: pd.Series,
                  lat2: pd.Series, lon2: pd.Series) -> pd.Series:
    """Vectorised haversine distance in kilometres."""
    R = 6_371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def _parse_hhmm_to_minutes(series: pd.Series) -> pd.Series:
    """Convert 'HH:MM' strings to total minutes since midnight."""
    def _convert(t):
        try:
            parts = str(t).strip().split(":")
            return int(parts[0]) * 60 + int(parts[1])
        except Exception:
            return np.nan
    return series.apply(_convert)


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds engineered and encoded columns.
    New columns:
        distance_km      – haversine distance restaurant → customer
        pickup_delay_min – minutes between order placed and rider pickup
        weather_n        – ordinal encoding (0=Sunny … 5=Cloudy)
        traffic_n        – ordinal encoding (0=Low … 3=Jam)
        vehicle_n        – ordinal encoding
        city_n           – ordinal encoding
        order_n          – ordinal encoding
        festival_n       – binary (0/1)
    """
    df = df.copy()

    # --- Haversine distance ------------------------------------------------
    df["distance_km"] = _haversine_km(
        df["Restaurant_latitude"],  df["Restaurant_longitude"],
        df["Delivery_location_latitude"], df["Delivery_location_longitude"],
    ).round(4)

    # --- Pickup delay -------------------------------------------------------
    order_min  = _parse_hhmm_to_minutes(df["Time_Orderd"])
    pickup_min = _parse_hhmm_to_minutes(df["Time_Order_picked"])
    # handle overnight wrap-around; clip outliers to 0–120 min
    delay = (pickup_min - order_min) % (24 * 60)
    df["pickup_delay_min"] = delay.clip(0, 120).round(2)

    # --- Categorical encodings ---------------------------------------------
    df["weather_n"]  = df["Weather_conditions"].map(WEATHER_MAP)
    df["traffic_n"]  = df["Road_traffic_density"].map(TRAFFIC_MAP)
    df["vehicle_n"]  = df["Type_of_vehicle"].map(VEHICLE_MAP)
    df["city_n"]     = df["City"].map(CITY_MAP)
    df["order_n"]    = df["Type_of_order"].map(ORDER_MAP)
    df["festival_n"] = df["Festival"].map(FESTIVAL_MAP)

    return df

## 4. Preprocessing (Clean + Split)

In [ ]:
def preprocess(df: pd.DataFrame):
    """
    Drop rows with nulls in required columns.
    Returns X_train, X_test, y_train, y_test, X_full, y_full.
    """
    required = [
        "Time_taken (min)", "Delivery_person_Age", "Delivery_person_Ratings",
        "multiple_deliveries", "Vehicle_condition", "distance_km",
        "pickup_delay_min", "weather_n", "traffic_n", "vehicle_n",
        "city_n", "order_n", "festival_n",
    ]
    before = len(df)
    df = df.dropna(subset=required).copy()
    dropped = before - len(df)
    print(f"[preprocess]  Dropped {dropped:,} rows with nulls → {len(df):,} remain")

    X = df[BASE_FEATURES]
    y = df["Time_taken (min)"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
    print(f"[preprocess]  Train {len(X_train):,} | Test {len(X_test):,}")
    return X_train, X_test, y_train, y_test, X, y

## 5. Model Training

In [ ]:
def train_random_forest(X_train, y_train) -> RandomForestRegressor:
    """Best overall model: R² 0.846, MAE 2.97 min."""
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def train_gradient_boosting(X_train, y_train) -> GradientBoostingRegressor:
    """Strong alternative: R² 0.836, MAE 3.06 min."""
    model = GradientBoostingRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42,
    )
    model.fit(X_train, y_train)
    return model


def train_ridge(X_train, y_train) -> Ridge:
    """Interpretable baseline: R² 0.557, MAE 5.01 min."""
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    return model

## 6. Evaluation

In [ ]:
def evaluate(name: str, model, X_test, y_test, X_full=None, y_full=None) -> dict:
    """Print and return evaluation metrics."""
    pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2   = r2_score(y_test, pred)

    cv_mae = None
    if X_full is not None and y_full is not None:
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        cv = cross_val_score(model, X_full, y_full,
                             cv=kf, scoring="neg_mean_absolute_error", n_jobs=-1)
        cv_mae = (-cv.mean(), cv.std())

    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(f"  MAE  : {mae:.4f} min")
    print(f"  RMSE : {rmse:.4f} min")
    print(f"  R²   : {r2:.4f}")
    if cv_mae:
        print(f"  CV-5 MAE: {cv_mae[0]:.4f} ± {cv_mae[1]:.4f}")

    return {"name": name, "mae": mae, "rmse": rmse, "r2": r2, "cv_mae": cv_mae}

## 7. Feature Importance

In [ ]:
def feature_importance_report(rf: RandomForestRegressor,
                               X_test, y_test,
                               save_plot: bool = True):
    """Print and optionally save a feature importance bar chart."""
    imp_rf = pd.Series(
        rf.feature_importances_,
        index=[FEATURE_LABELS.get(f, f) for f in BASE_FEATURES]
    ).sort_values(ascending=False)

    print("\n── Feature Importance (Random Forest) ──────────────────")
    for feat, val in imp_rf.items():
        bar = "█" * int(val * 200)
        print(f"  {feat:<25} {val*100:5.2f}%  {bar}")

    # Permutation importance on test set (model-agnostic)
    perm = permutation_importance(rf, X_test, y_test,
                                  n_repeats=10, random_state=42, n_jobs=-1)
    imp_perm = pd.Series(
        perm.importances_mean,
        index=[FEATURE_LABELS.get(f, f) for f in BASE_FEATURES]
    ).sort_values(ascending=False)

    print("\n── Permutation Importance (test set) ───────────────────")
    for feat, val in imp_perm.items():
        print(f"  {feat:<25} Δ MAE {val:+.4f} min")

    if save_plot:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        imp_rf.sort_values().plot(kind="barh", ax=ax1, color="#378ADD")
        ax1.set_title("RF Feature Importance", fontsize=13)
        ax1.set_xlabel("Importance (%)")
        ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v*100:.0f}%"))

        imp_perm.sort_values().plot(kind="barh", ax=ax2, color="#1D9E75")
        ax2.set_title("Permutation Importance (Δ MAE)", fontsize=13)
        ax2.set_xlabel("Δ MAE (min)")

        plt.tight_layout()
        plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
        print("\n[plot] Saved → feature_importance.png")
        plt.close()

    return imp_rf, imp_perm

## 8. Ridge Formula (Interpretable)

In [ ]:
def print_ridge_formula(ridge: Ridge):
    """Print the full linear equation with real coefficients."""
    print("\n── Ridge Regression Formula ────────────────────────────")
    print(f"  pred = {ridge.intercept_:.4f}")
    for feat, coef in zip(BASE_FEATURES, ridge.coef_):
        sign = "+" if coef >= 0 else "−"
        label = FEATURE_LABELS.get(feat, feat)
        print(f"       {sign} {abs(coef):.4f} × {label}")

## 9. Prediction Interface

In [ ]:
def predict_single(
    model: RandomForestRegressor,
    *,
    weather: str        = "Sunny",
    traffic: str        = "Low",
    vehicle: str        = "scooter",
    city: str           = "Urban",
    order_type: str     = "Meal",
    festival: str       = "No",
    rider_age: float    = 29.6,
    rider_rating: float = 4.64,
    vehicle_condition: int = 1,
    multiple_deliveries: int = 0,
    distance_km: float  = 5.0,
    pickup_delay_min: float = 10.0,
) -> dict:
    """
    Predict delivery time for a single order.

    Parameters
    ----------
    weather            : 'Sunny' | 'Windy' | 'Sandstorms' | 'Stormy' | 'Fog' | 'Cloudy'
    traffic            : 'Low' | 'Medium' | 'High' | 'Jam'
    vehicle            : 'electric_scooter' | 'scooter' | 'bicycle' | 'motorcycle'
    city               : 'Urban' | 'Metropolitian' | 'Semi-Urban'
    order_type         : 'Snack' | 'Drinks' | 'Meal' | 'Buffet'
    festival           : 'No' | 'Yes'
    rider_age          : age in years (15–50)
    rider_rating       : 1.0–6.0
    vehicle_condition  : 0 (worst) – 3 (best)
    multiple_deliveries: 0 – 3
    distance_km        : haversine distance in km
    pickup_delay_min   : minutes between order placed and pickup (0–120)

    Returns
    -------
    dict with keys: predicted_min, lower_80, upper_80, sla_status, confidence
    """
    row = {
        "Delivery_person_Age":     rider_age,
        "Delivery_person_Ratings": rider_rating,
        "Vehicle_condition":       vehicle_condition,
        "multiple_deliveries":     multiple_deliveries,
        "weather_n":  WEATHER_MAP.get(weather, 0),
        "traffic_n":  TRAFFIC_MAP.get(traffic, 0),
        "vehicle_n":  VEHICLE_MAP.get(vehicle, 1),
        "city_n":     CITY_MAP.get(city, 0),
        "order_n":    ORDER_MAP.get(order_type, 1),
        "festival_n": FESTIVAL_MAP.get(festival, 0),
        "distance_km":       distance_km,
        "pickup_delay_min":  pickup_delay_min,
    }
    X = pd.DataFrame([row])[BASE_FEATURES]
    pred = float(model.predict(X)[0])

    # 80% confidence interval from tree variance
    tree_preds = np.array([t.predict(X)[0] for t in model.estimators_])
    std = tree_preds.std()
    lo  = round(max(10.0, pred - 1.28 * std), 1)
    hi  = round(min(60.0, pred + 1.28 * std), 1)
    conf = round(float(1 - std / (tree_preds.mean() + 1e-9)), 3)

    sla = "On time" if pred <= 30 else ("At risk" if pred <= 40 else "Delayed")

    return {
        "predicted_min": round(pred, 1),
        "lower_80":      lo,
        "upper_80":      hi,
        "sla_status":    sla,
        "confidence":    conf,
    }


def predict_batch(model: RandomForestRegressor, df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Run prediction on a pre-engineered DataFrame that already has all BASE_FEATURES.
    Returns the input DataFrame with two new columns: pred_min, sla_status.
    """
    X = df_raw[BASE_FEATURES]
    df_out = df_raw.copy()
    df_out["pred_min"]   = model.predict(X).round(1)
    df_out["sla_status"] = df_out["pred_min"].apply(
        lambda v: "On time" if v <= 30 else ("At risk" if v <= 40 else "Delayed")
    )
    return df_out

## 10. Model Persistence

In [ ]:
def save_models(models: dict, path: str = "zomato_models.joblib"):
    joblib.dump(models, path, compress=3)
    size_mb = os.path.getsize(path) / 1_048_576
    print(f"\n[save] Models saved → {path}  ({size_mb:.1f} MB)")


def load_models(path: str = "zomato_models.joblib") -> dict:
    bundle = joblib.load(path)
    print(f"[load] Models loaded from {path}")
    return bundle

## 11. Residual Analysis Plot

In [ ]:
def plot_residuals(model, X_test, y_test, name: str = "Random Forest"):
    pred = model.predict(X_test)
    residuals = y_test.values - pred

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Predicted vs Actual
    axes[0].scatter(y_test, pred, alpha=0.3, s=10, color="#378ADD")
    lim = [min(y_test.min(), pred.min()) - 2, max(y_test.max(), pred.max()) + 2]
    axes[0].plot(lim, lim, "r--", lw=1)
    axes[0].set_xlabel("Actual (min)")
    axes[0].set_ylabel("Predicted (min)")
    axes[0].set_title(f"{name} — Actual vs Predicted")

    # Residuals vs Predicted
    axes[1].scatter(pred, residuals, alpha=0.3, s=10, color="#7F77DD")
    axes[1].axhline(0, color="red", lw=1, ls="--")
    axes[1].set_xlabel("Predicted (min)")
    axes[1].set_ylabel("Residual (min)")
    axes[1].set_title("Residuals vs Predicted")

    # Residual distribution
    axes[2].hist(residuals, bins=60, color="#1D9E75", edgecolor="none", alpha=0.85)
    axes[2].axvline(0, color="red", lw=1, ls="--")
    axes[2].set_xlabel("Residual (min)")
    axes[2].set_ylabel("Count")
    axes[2].set_title(f"Residual distribution  (σ={residuals.std():.2f})")

    plt.tight_layout()
    plt.savefig("residuals.png", dpi=150, bbox_inches="tight")
    print("[plot] Saved → residuals.png")
    plt.close()

## 12. Main Pipeline

In [ ]:
def main(data_path: str = "app/data/delivery.csv"):
    print("=" * 60)
    print("  Zomato Delivery Time — ML Pipeline")
    print("=" * 60)

    # ── Load ────────────────────────────────────────────────────────────────
    df_raw = load_data(data_path)

    # ── Feature engineering ─────────────────────────────────────────────────
    df = engineer_features(df_raw)

    # ── Preprocess + split ──────────────────────────────────────────────────
    X_train, X_test, y_train, y_test, X_full, y_full = preprocess(df)

    # ── Train ───────────────────────────────────────────────────────────────
    print("\n[train] Fitting Random Forest …")
    rf = train_random_forest(X_train, y_train)

    print("[train] Fitting Gradient Boosting …")
    gbr = train_gradient_boosting(X_train, y_train)

    print("[train] Fitting Ridge …")
    ridge = train_ridge(X_train, y_train)

    # ── Evaluate ────────────────────────────────────────────────────────────
    results = {}
    results["rf"]    = evaluate("Random Forest",      rf,    X_test, y_test, X_full, y_full)
    results["gbr"]   = evaluate("Gradient Boosting",  gbr,   X_test, y_test)
    results["ridge"] = evaluate("Ridge Regression",   ridge, X_test, y_test)

    # ── Feature importance ──────────────────────────────────────────────────
    feature_importance_report(rf, X_test, y_test, save_plot=True)

    # ── Ridge formula ────────────────────────────────────────────────────────
    print_ridge_formula(ridge)

    # ── Residual plots ───────────────────────────────────────────────────────
    plot_residuals(rf, X_test, y_test)

    # ── Demo prediction ──────────────────────────────────────────────────────
    print("\n── Single-order prediction demo ─────────────────────────")
    demo = predict_single(
        rf,
        weather="Cloudy",
        traffic="Jam",
        vehicle="motorcycle",
        city="Metropolitian",
        festival="No",
        rider_age=32,
        rider_rating=4.2,
        vehicle_condition=1,
        multiple_deliveries=1,
        distance_km=7.5,
        pickup_delay_min=12.0,
    )
    print(f"  Predicted      : {demo['predicted_min']} min")
    print(f"  80% CI         : {demo['lower_80']} – {demo['upper_80']} min")
    print(f"  SLA status     : {demo['sla_status']}")
    print(f"  Confidence     : {demo['confidence']*100:.1f}%")

    # ── Save ─────────────────────────────────────────────────────────────────
    save_models({"rf": rf, "gbr": gbr, "ridge": ridge,
                 "feature_names": BASE_FEATURES,
                 "metrics": results}, "zomato_models.joblib")

    print("\n✓ Pipeline complete.")
    return rf, gbr, ridge, results

## 13. Run Pipeline

In [ ]:
import sys
data_path = sys.argv[1] if len(sys.argv) > 1 else "app/data/delivery.csv"
rf, gbr, ridge, results = main(data_path)